# Phase 4A -- TaViT: Temporal-Aware Trajectory Modeling

> **Purpose**: Learn temporal dependencies across longitudinal MRI scans using a lightweight
> Transformer on pre-extracted SwinUNETR hybrid embeddings (4873-D).
>
> **Architecture**: Linear projection -> Continuous-time PE -> 4-layer Causal Transformer -> Trajectory embedding (256-D)
>
> **Training**: (A) Self-supervised chronological ordering, (B) Supervised progression regression
>
> **References**:
> 1. Tak et al. "Longitudinal Risk Prediction for Pediatric Glioma" (NEJM AI 2025)
> 2. Holste et al. "LTSA" (npj Digital Medicine 2024)
> 3. Nolte et al. "HCCNet" (arXiv 2025)


In [ ]:
import numpy as np
import json as _json
import math, os, warnings, time
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from scipy.stats import spearmanr, kendalltau

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

# -- Config --
INPUT_DIM   = 4873
PROJ_DIM    = 256
N_HEADS     = 8
N_LAYERS    = 4
DROPOUT     = 0.2
MAX_SEQ_LEN = 8
LR_SSL      = 1e-4
LR_FINE     = 5e-5
EPOCHS_SSL  = 60
EPOCHS_FINE = 40
BATCH_SIZE  = 32
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

OUTPUT_ROOT = Path("/kaggle/working/tavit_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Config: proj={PROJ_DIM}, heads={N_HEADS}, layers={N_LAYERS}, max_seq={MAX_SEQ_LEN}")

# -- Search for files --
SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working")]

def find_file(names):
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for f in root.rglob("*"):
            for name in names:
                if name in f.name:
                    return f
    return None

# Load hybrid embeddings
emb_path = find_file(["vit_swinunetr_embeddings_v5_hybrid.npz",
                       "vit_swinunetr_embeddings_v5_noglobal.npz"])
assert emb_path is not None, "Hybrid embeddings NPZ not found!"
npz = np.load(emb_path, allow_pickle=True)
all_npz_keys = sorted(npz.files)
print(f"\nLoaded: {emb_path.name}")
print(f"  NPZ keys ({len(all_npz_keys)}): {all_npz_keys[:6]}")

# -- Detect NPZ format --
# Format A (per-scan keys): each key looks like "BraTS-GLI-00005__t100"
# Format B (batch format): keys are "embeddings", "scan_ids", etc.
def is_scan_key(k):
    return "__" in k and any(k.split("__")[1].startswith(p) for p in ["t", "p"])

format_a_keys = [k for k in all_npz_keys if is_scan_key(k)]
is_format_a = len(format_a_keys) >= len(all_npz_keys) * 0.5

if is_format_a:
    print(f"  Format: PER-SCAN keys ({len(format_a_keys)} scans)")
    all_keys = format_a_keys
    scan_emb_dict = {k: npz[k].astype(np.float32) for k in all_keys}
else:
    # Format B: find embeddings matrix and scan_ids array
    print(f"  Format: BATCH arrays")
    emb_key = next((k for k in all_npz_keys if npz[k].ndim == 2), None)
    id_key  = next((k for k in all_npz_keys if npz[k].ndim == 1
                    and npz[k].dtype.kind in ['U', 'S', 'O']), None)

    if emb_key is None:
        # Try: largest 2D array
        emb_key = max(all_npz_keys, key=lambda k: npz[k].size)
    if id_key is None:
        # Try: any 1D array that isn't the embeddings
        id_key = next((k for k in all_npz_keys if k != emb_key and npz[k].ndim == 1), None)

    emb_matrix = npz[emb_key].astype(np.float32)
    print(f"  Embeddings array: '{emb_key}' shape={emb_matrix.shape}")

    # Find patient_ids and timepoints arrays
    pid_key = next((k for k in all_npz_keys if 'patient' in k.lower() or 'pid' in k.lower()), id_key)
    tp_key  = next((k for k in all_npz_keys if 'time' in k.lower() or 'tp' == k.lower()
                    or k.lower() in ['timepoints', 'timepoint', 'tps']), None)

    raw_pids = [str(s) for s in npz[pid_key]] if pid_key else [f"scan_{i:05d}" for i in range(len(emb_matrix))]
    raw_tps  = list(npz[tp_key]) if tp_key else list(range(len(emb_matrix)))

    print(f"  Patient IDs array: '{pid_key}' n={len(raw_pids)} | example: {raw_pids[:2]}")
    if tp_key:
        print(f"  Timepoints array:  '{tp_key}' n={len(raw_tps)} | example: {raw_tps[:4]}")
    else:
        print(f"  No timepoints array -- using row index as timepoint")

    # Build per-scan lookup: (patient_id, timepoint) -> embedding
    # Note: use a composite key to avoid deduplication
    all_keys = [f"{raw_pids[i]}__t{int(raw_tps[i])}" for i in range(len(emb_matrix))]
    scan_emb_dict = {all_keys[i]: emb_matrix[i] for i in range(len(emb_matrix))}
    print(f"  Composite keys built: {len(scan_emb_dict)} | example: {all_keys[:2]}")

actual_dim = next(iter(scan_emb_dict.values())).shape[0]
print(f"  Embedding dim: {actual_dim} | Total scans: {len(scan_emb_dict)}")
if actual_dim != INPUT_DIM:
    print(f"  Adjusting INPUT_DIM: {INPUT_DIM} -> {actual_dim}")
    INPUT_DIM = actual_dim

# Load tumor volumes
vol_path = find_file(["tumor_volumes.csv"])
if vol_path:
    vol_df = pd.read_csv(vol_path)
    print(f"  Tumor volumes: {len(vol_df)} rows from {vol_path.name}")
else:
    vol_df = None
    print("  WARNING: No tumor_volumes.csv found")

# -- Build per-patient temporal sequences --
patient_seqs = defaultdict(list)
for key in all_keys:
    emb = scan_emb_dict[key]
    parts = key.split("__")
    if len(parts) == 2:
        # Standard format: "BraTS-GLI-00005__t100"
        pid, tp_str = parts
        tp = int("".join(filter(str.isdigit, tp_str)) or "0")
    elif len(parts) == 1 and "_" in key:
        # Fallback: try "PATIENTID_TIMEPOINT"
        tokens = key.rsplit("_", 1)
        pid = tokens[0]
        tp = int("".join(filter(str.isdigit, tokens[1])) or "0") if len(tokens) > 1 else 0
    else:
        continue
    patient_seqs[pid].append((tp, emb, key))

for pid in patient_seqs:
    patient_seqs[pid].sort(key=lambda x: x[0])

longitudinal = {pid: seqs for pid, seqs in patient_seqs.items() if len(seqs) >= 2}
single_visit = {pid: seqs for pid, seqs in patient_seqs.items() if len(seqs) < 2}

seq_lens = [len(v) for v in longitudinal.values()]
print(f"\nTemporal sequences:")
print(f"  Longitudinal patients (>=2 scans): {len(longitudinal)}")
print(f"  Single-visit patients: {len(single_visit)}")
if not seq_lens:
    print("  WARNING: No longitudinal patients found!")
    print(f"  Sample keys: {list(scan_emb_dict.keys())[:5]}")
    print("  Check that the NPZ uses '__' separator: 'PATIENTID__tTIMEPOINT'")
else:
    print(f"  Seq lengths: min={min(seq_lens)} max={max(seq_lens)} "
          f"mean={np.mean(seq_lens):.1f} median={np.median(seq_lens):.0f}")
    for sl in sorted(set(seq_lens)):
        count = seq_lens.count(sl)
        print(f"    {sl} scans: {count} patients ({100*count/len(longitudinal):.0f}%)")

assert len(longitudinal) >= 10, (
    f"Only {len(longitudinal)} longitudinal patients found. "
    f"Check NPZ key format -- expected 'PATIENTID__tTIMEPOINT'. "
    f"Sample keys: {list(scan_emb_dict.keys())[:5]}"
)

# -- Train/Val/Test split --
pids = sorted(longitudinal.keys())
np.random.shuffle(pids)
n = len(pids)
n_test = max(50, int(n * 0.15))
n_val  = max(40, int(n * 0.15))
n_train = n - n_test - n_val

pids_train = pids[:n_train]
pids_val   = pids[n_train:n_train+n_val]
pids_test  = pids[n_train+n_val:]
print(f"\nSplit: train={len(pids_train)} | val={len(pids_val)} | test={len(pids_test)}")


In [ ]:
# ============================================================
# TaViT ARCHITECTURE
# ============================================================

class ContinuousTimePositionalEncoding(nn.Module):
    """Sinusoidal PE over elapsed days since baseline (from LTSA).
    Handles irregular scan intervals unlike discrete position indices."""

    def __init__(self, d_model, max_days=3650):
        super().__init__()
        self.d_model = d_model
        self.max_days = max_days

    def forward(self, days_since_baseline):
        B, S = days_since_baseline.shape
        pe = torch.zeros(B, S, self.d_model, device=days_since_baseline.device)
        t = days_since_baseline.float().unsqueeze(-1) / self.max_days
        div_term = torch.exp(
            torch.arange(0, self.d_model, 2, device=days_since_baseline.device).float()
            * (-math.log(10000.0) / self.d_model)
        )
        pe[:, :, 0::2] = torch.sin(t * div_term * self.max_days)
        pe[:, :, 1::2] = torch.cos(t * div_term * self.max_days)
        return pe


class TaViT(nn.Module):
    """Temporal-aware Vision Transformer for longitudinal trajectory modeling.
    Takes a sequence of per-scan embeddings, produces a trajectory embedding."""

    def __init__(self, input_dim, proj_dim=256, n_heads=8, n_layers=4,
                 dropout=0.2, max_seq_len=8):
        super().__init__()
        self.proj_dim = proj_dim
        self.max_seq_len = max_seq_len

        # Step 1: Linear projection
        self.input_proj = nn.Sequential(
            nn.Linear(input_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # Step 2: Continuous-time positional encoding
        self.time_pe = ContinuousTimePositionalEncoding(proj_dim)

        # Step 3: Learnable [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, proj_dim) * 0.02)

        # Step 4: Causal Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=proj_dim, nhead=n_heads,
            dim_feedforward=proj_dim * 4, dropout=dropout,
            activation="gelu", batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # Step 5: Output projection
        self.trajectory_head = nn.Sequential(
            nn.LayerNorm(proj_dim),
            nn.Linear(proj_dim, proj_dim),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, emb_seq, days_seq, padding_mask=None):
        """
        Args:
            emb_seq:      (B, S, input_dim) sequence of scan embeddings
            days_seq:     (B, S) days since baseline for each scan
            padding_mask: (B, S) True for padded positions
        Returns:
            trajectory:   (B, proj_dim) trajectory embedding
            token_out:    (B, S+1, proj_dim) all token outputs
        """
        B, S, _ = emb_seq.shape
        x = self.input_proj(emb_seq)
        x = x + self.time_pe(days_seq)

        # Prepend CLS token
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)

        # Causal mask: each position attends to itself and all previous
        causal_mask = torch.triu(
            torch.ones(S + 1, S + 1, device=x.device), diagonal=1
        ).bool()

        # Extend padding mask for CLS token
        if padding_mask is not None:
            cls_pad = torch.zeros(B, 1, dtype=torch.bool, device=x.device)
            src_key_padding = torch.cat([cls_pad, padding_mask], dim=1)
        else:
            src_key_padding = None

        token_out = self.transformer(x, mask=causal_mask,
                                     src_key_padding_mask=src_key_padding)
        cls_out = token_out[:, 0, :]
        trajectory = self.trajectory_head(cls_out)
        return trajectory, token_out


class SSLOrderingHead(nn.Module):
    """Binary classifier: is this sequence in chronological order?"""
    def __init__(self, proj_dim):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(proj_dim, 128), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(128, 1),
        )
    def forward(self, trajectory):
        return self.head(trajectory).squeeze(-1)


class ProgressionHead(nn.Module):
    """Regression head for volume change prediction."""
    def __init__(self, proj_dim):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(proj_dim, 128), nn.GELU(),
            nn.Dropout(0.2), nn.Linear(128, 1),
        )
    def forward(self, trajectory):
        return self.head(trajectory).squeeze(-1)


# -- Instantiate --
model = TaViT(input_dim=INPUT_DIM, proj_dim=PROJ_DIM, n_heads=N_HEADS,
              n_layers=N_LAYERS, dropout=DROPOUT, max_seq_len=MAX_SEQ_LEN).to(DEVICE)
ssl_head = SSLOrderingHead(PROJ_DIM).to(DEVICE)
prog_head = ProgressionHead(PROJ_DIM).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
n_ssl = sum(p.numel() for p in ssl_head.parameters())
n_prog = sum(p.numel() for p in prog_head.parameters())
print(f"TaViT parameters:       {n_params:,}")
print(f"SSL ordering head:      {n_ssl:,}")
print(f"Progression head:       {n_prog:,}")
print(f"Total:                  {n_params + n_ssl + n_prog:,}")


In [ ]:
# ============================================================
# DATASETS
# ============================================================

def get_volume_for_scan(vdf, pid, tp):
    """Get tumor volume for a specific scan."""
    if vdf is None:
        return None
    pid_short = pid.split("-")[-1] if "-" in pid else pid
    tp_int = int(str(tp).replace("t", "").replace("p", ""))

    for _, row in vdf.iterrows():
        row_pid = str(row.get("patient_id", row.get("PatientID", "")))
        row_tp = row.get("timepoint", row.get("Timepoint", -1))
        if pid_short in row_pid or pid in row_pid:
            if int(row_tp) == tp_int:
                for vcol in ["total_volume_mm3", "whole_tumor_vol", "WT_vol", "volume"]:
                    if vcol in row.index and pd.notna(row[vcol]):
                        return float(row[vcol])
    return None

# Pre-build a fast lookup dict for volumes
vol_lookup = {}
if vol_df is not None:
    pid_col = None
    tp_col = None
    vol_col = None
    for c in ["patient_id", "PatientID", "pid"]:
        if c in vol_df.columns:
            pid_col = c; break
    for c in ["timepoint", "Timepoint", "tp"]:
        if c in vol_df.columns:
            tp_col = c; break
    for c in ["total_volume_mm3", "whole_tumor_vol", "WT_vol", "volume"]:
        if c in vol_df.columns:
            vol_col = c; break

    if pid_col and tp_col and vol_col:
        for _, row in vol_df.iterrows():
            key = (str(row[pid_col]), int(row[tp_col]))
            vol_lookup[key] = float(row[vol_col])
        print(f"Volume lookup built: {len(vol_lookup)} entries")
        print(f"  Columns used: pid={pid_col}, tp={tp_col}, vol={vol_col}")

def fast_vol_lookup(pid, tp):
    tp_int = int(str(tp).replace("t", "").replace("p", ""))
    # Try exact match
    for key_pid in [pid, pid.split("-")[-1]]:
        if (key_pid, tp_int) in vol_lookup:
            return vol_lookup[(key_pid, tp_int)]
    # Try partial match
    for (k_pid, k_tp), v in vol_lookup.items():
        if tp_int == k_tp and (pid in k_pid or k_pid in pid):
            return v
    return None


class TemporalSSLDataset(Dataset):
    """Dataset for chronological ordering SSL."""

    def __init__(self, patient_seqs, pids, max_seq_len=8):
        self.sequences = []
        self.max_seq_len = max_seq_len
        for pid in pids:
            seqs = patient_seqs[pid]
            embs = np.stack([s[1] for s in seqs])
            tps = np.array([s[0] for s in seqs])
            days = (tps - tps[0]).astype(np.float32) * 90.0
            self.sequences.append((embs, days, pid))

    def __len__(self):
        return len(self.sequences) * 2

    def __getitem__(self, idx):
        seq_idx = idx // 2
        is_positive = (idx % 2 == 0)
        embs, days, pid = self.sequences[seq_idx]
        S = len(embs)

        if is_positive:
            order = np.arange(S)
            label = 1.0
        else:
            order = np.random.permutation(S)
            while S > 1 and np.all(order == np.arange(S)):
                order = np.random.permutation(S)
            label = 0.0

        embs_ordered = embs[order]
        days_ordered = days[order]
        pad_len = self.max_seq_len - S

        if pad_len > 0:
            embs_p = np.concatenate([embs_ordered,
                np.zeros((pad_len, embs.shape[1]), dtype=np.float32)])
            days_p = np.concatenate([days_ordered,
                np.zeros(pad_len, dtype=np.float32)])
            mask = np.array([False] * S + [True] * pad_len)
        else:
            embs_p = embs_ordered[:self.max_seq_len]
            days_p = days_ordered[:self.max_seq_len]
            mask = np.array([False] * self.max_seq_len)

        return (torch.tensor(embs_p), torch.tensor(days_p),
                torch.tensor(label), torch.tensor(mask))


class TemporalProgressionDataset(Dataset):
    """Dataset for supervised progression regression."""

    def __init__(self, patient_seqs, pids, max_seq_len=8):
        self.sequences = []
        self.max_seq_len = max_seq_len
        skipped = 0
        for pid in pids:
            seqs = patient_seqs[pid]
            v_first = fast_vol_lookup(pid, seqs[0][0])
            v_last = fast_vol_lookup(pid, seqs[-1][0])
            if v_first is None or v_last is None or v_first < 1.0:
                skipped += 1
                continue
            delta = (v_last - v_first) / v_first
            embs = np.stack([s[1] for s in seqs])
            tps = np.array([s[0] for s in seqs])
            days = (tps - tps[0]).astype(np.float32) * 90.0
            self.sequences.append((embs, days, delta, pid))
        print(f"  Progression dataset: {len(self.sequences)} patients "
              f"({skipped} skipped, no volume data)")

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        embs, days, delta, pid = self.sequences[idx]
        S = len(embs)
        pad_len = self.max_seq_len - S
        if pad_len > 0:
            embs_p = np.concatenate([embs,
                np.zeros((pad_len, embs.shape[1]), dtype=np.float32)])
            days_p = np.concatenate([days, np.zeros(pad_len, dtype=np.float32)])
            mask = np.array([False] * S + [True] * pad_len)
        else:
            embs_p = embs[:self.max_seq_len]
            days_p = days[:self.max_seq_len]
            mask = np.array([False] * self.max_seq_len)
        return (torch.tensor(embs_p), torch.tensor(days_p),
                torch.tensor(delta, dtype=torch.float32), torch.tensor(mask))


ssl_train = TemporalSSLDataset(longitudinal, pids_train, MAX_SEQ_LEN)
ssl_val   = TemporalSSLDataset(longitudinal, pids_val, MAX_SEQ_LEN)
print(f"SSL datasets: train={len(ssl_train)} | val={len(ssl_val)} samples")

ssl_train_loader = DataLoader(ssl_train, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, pin_memory=True)
ssl_val_loader   = DataLoader(ssl_val, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, pin_memory=True)


In [ ]:
# ============================================================
# PHASE A: SELF-SUPERVISED CHRONOLOGICAL ORDERING
# (Tak et al. NEJM AI 2025)
# ============================================================
print("=" * 60)
print("  PHASE A: Self-Supervised Chronological Ordering")
print("=" * 60)

optimizer_ssl = torch.optim.AdamW(
    list(model.parameters()) + list(ssl_head.parameters()),
    lr=LR_SSL, weight_decay=0.01
)
scheduler_ssl = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ssl, T_max=EPOCHS_SSL)
criterion_ssl = nn.BCEWithLogitsLoss()

best_val_acc = 0
best_epoch = 0

for epoch in range(EPOCHS_SSL):
    model.train(); ssl_head.train()
    total_loss = 0; total_correct = 0; total_samples = 0

    for embs, days, labels, masks in ssl_train_loader:
        embs = embs.to(DEVICE); days = days.to(DEVICE)
        labels = labels.to(DEVICE); masks = masks.to(DEVICE)

        trajectory, _ = model(embs, days, masks)
        logits = ssl_head(trajectory)
        loss = criterion_ssl(logits, labels)

        optimizer_ssl.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer_ssl.step()

        total_loss += loss.item() * len(labels)
        preds = (torch.sigmoid(logits) > 0.5).float()
        total_correct += (preds == labels).sum().item()
        total_samples += len(labels)

    train_loss = total_loss / total_samples
    train_acc = total_correct / total_samples

    # -- Validate --
    model.eval(); ssl_head.eval()
    val_correct = 0; val_total = 0
    with torch.no_grad():
        for embs, days, labels, masks in ssl_val_loader:
            embs = embs.to(DEVICE); days = days.to(DEVICE)
            labels = labels.to(DEVICE); masks = masks.to(DEVICE)
            trajectory, _ = model(embs, days, masks)
            logits = ssl_head(trajectory)
            preds = (torch.sigmoid(logits) > 0.5).float()
            val_correct += (preds == labels).sum().item()
            val_total += len(labels)

    val_acc = val_correct / val_total
    scheduler_ssl.step()

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save({
            "model": model.state_dict(),
            "ssl_head": ssl_head.state_dict(),
            "epoch": epoch, "val_acc": val_acc,
        }, OUTPUT_ROOT / "tavit_ssl_best.pt")

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(f"  Epoch {epoch+1:3d}/{EPOCHS_SSL} | "
              f"loss={train_loss:.4f} | train_acc={train_acc:.3f} | "
              f"val_acc={val_acc:.3f} | best={best_val_acc:.3f} (e{best_epoch+1})")

print(f"\n  SSL pretraining complete")
print(f"  Best val accuracy: {best_val_acc:.3f} at epoch {best_epoch+1}")
print(f"  (random baseline = 0.500)")

ckpt = torch.load(OUTPUT_ROOT / "tavit_ssl_best.pt", map_location=DEVICE)
model.load_state_dict(ckpt["model"])
print(f"  Loaded best SSL checkpoint (epoch {ckpt['epoch']+1})")


In [ ]:
# ============================================================
# PHASE B: SUPERVISED PROGRESSION REGRESSION
# ============================================================
print("=" * 60)
print("  PHASE B: Supervised Progression Regression")
print("=" * 60)

if vol_df is not None and len(vol_lookup) > 0:
    prog_train = TemporalProgressionDataset(longitudinal, pids_train, MAX_SEQ_LEN)
    prog_val   = TemporalProgressionDataset(longitudinal, pids_val, MAX_SEQ_LEN)

    if len(prog_train) > 10 and len(prog_val) > 5:
        prog_train_loader = DataLoader(prog_train, batch_size=BATCH_SIZE, shuffle=True,
                                       num_workers=2, pin_memory=True)
        prog_val_loader   = DataLoader(prog_val, batch_size=BATCH_SIZE, shuffle=False,
                                       num_workers=2, pin_memory=True)

        optimizer_prog = torch.optim.AdamW(
            list(model.parameters()) + list(prog_head.parameters()),
            lr=LR_FINE, weight_decay=0.01
        )
        scheduler_prog = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer_prog, T_max=EPOCHS_FINE)
        criterion_prog = nn.SmoothL1Loss()
        best_val_mae = float("inf")

        for epoch in range(EPOCHS_FINE):
            model.train(); prog_head.train()
            total_loss = 0; n = 0

            for embs, days, delta_vol, masks in prog_train_loader:
                embs = embs.to(DEVICE); days = days.to(DEVICE)
                delta_vol = delta_vol.to(DEVICE); masks = masks.to(DEVICE)

                trajectory, _ = model(embs, days, masks)
                pred = prog_head(trajectory)
                loss = criterion_prog(pred, delta_vol)

                optimizer_prog.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer_prog.step()
                total_loss += loss.item() * len(delta_vol)
                n += len(delta_vol)

            train_loss = total_loss / n

            model.eval(); prog_head.eval()
            val_preds = []; val_labels = []
            with torch.no_grad():
                for embs, days, delta_vol, masks in prog_val_loader:
                    embs = embs.to(DEVICE); days = days.to(DEVICE)
                    delta_vol = delta_vol.to(DEVICE); masks = masks.to(DEVICE)
                    trajectory, _ = model(embs, days, masks)
                    pred = prog_head(trajectory)
                    val_preds.extend(pred.cpu().numpy())
                    val_labels.extend(delta_vol.cpu().numpy())

            val_preds = np.array(val_preds)
            val_labels = np.array(val_labels)
            val_mae = np.mean(np.abs(val_preds - val_labels))
            val_rho, _ = spearmanr(val_preds, val_labels)

            if val_mae < best_val_mae:
                best_val_mae = val_mae
                torch.save({
                    "model": model.state_dict(),
                    "prog_head": prog_head.state_dict(),
                    "epoch": epoch, "val_mae": val_mae, "val_rho": val_rho,
                }, OUTPUT_ROOT / "tavit_finetuned_best.pt")

            scheduler_prog.step()

            if (epoch + 1) % 10 == 0 or epoch == 0:
                print(f"  Epoch {epoch+1:3d}/{EPOCHS_FINE} | loss={train_loss:.4f} | "
                      f"val_MAE={val_mae:.4f} | val_Spearman={val_rho:.3f}")

        ckpt = torch.load(OUTPUT_ROOT / "tavit_finetuned_best.pt", map_location=DEVICE)
        model.load_state_dict(ckpt["model"])
        print(f"\n  Fine-tuning complete")
        print(f"  Best val MAE: {ckpt['val_mae']:.4f} | Spearman: {ckpt['val_rho']:.3f}")
    else:
        print(f"  Not enough data for fine-tuning (train={len(prog_train)}, val={len(prog_val)})")
        print(f"  Using SSL-only model")
else:
    print("  Skipping fine-tuning (no tumor_volumes.csv)")
    print("  Using SSL-only model for embedding extraction")


In [ ]:
# ============================================================
# EXTRACT TRAJECTORY EMBEDDINGS (per patient)
# ============================================================
print("=" * 60)
print("  EXTRACTING TRAJECTORY EMBEDDINGS")
print("=" * 60)

model.eval()
trajectory_embs = {}
trajectory_meta = {}

with torch.no_grad():
    for pid, seqs in longitudinal.items():
        embs = np.stack([s[1] for s in seqs]).astype(np.float32)
        tps = np.array([s[0] for s in seqs])
        days = (tps - tps[0]).astype(np.float32) * 90.0
        S = len(embs)
        pad_len = MAX_SEQ_LEN - S

        if pad_len > 0:
            embs_p = np.concatenate([embs,
                np.zeros((pad_len, embs.shape[1]), dtype=np.float32)])
            days_p = np.concatenate([days, np.zeros(pad_len, dtype=np.float32)])
            mask = np.array([False] * S + [True] * pad_len)
        else:
            embs_p = embs[:MAX_SEQ_LEN]
            days_p = days[:MAX_SEQ_LEN]
            mask = np.array([False] * MAX_SEQ_LEN)

        embs_t = torch.tensor(embs_p).unsqueeze(0).to(DEVICE)
        days_t = torch.tensor(days_p).unsqueeze(0).to(DEVICE)
        mask_t = torch.tensor(mask).unsqueeze(0).to(DEVICE)

        trajectory, _ = model(embs_t, days_t, mask_t)
        trajectory_embs[pid] = trajectory.cpu().numpy().squeeze()
        trajectory_meta[pid] = {
            "n_scans": S,
            "timepoints": tps.tolist(),
            "days_span": float(days[-1] - days[0]),
            "split": ("train" if pid in pids_train
                      else "val" if pid in pids_val else "test"),
        }

print(f"  Extracted: {len(trajectory_embs)} patients")
print(f"  Embedding dim: {list(trajectory_embs.values())[0].shape[0]}")

np.savez_compressed(OUTPUT_ROOT / "tavit_trajectory_embeddings.npz",
                    **trajectory_embs)
with open(OUTPUT_ROOT / "tavit_trajectory_meta.json", "w") as f:
    _json.dump(trajectory_meta, f, indent=2)

torch.save({
    "model": model.state_dict(),
    "config": {"input_dim": INPUT_DIM, "proj_dim": PROJ_DIM,
               "n_heads": N_HEADS, "n_layers": N_LAYERS,
               "dropout": DROPOUT, "max_seq_len": MAX_SEQ_LEN},
}, OUTPUT_ROOT / "tavit_model_final.pt")

print(f"\n  Saved: tavit_trajectory_embeddings.npz")
print(f"  Saved: tavit_trajectory_meta.json")
print(f"  Saved: tavit_model_final.pt")


In [ ]:
# ============================================================
# QUICK TEMPORAL VALIDATION (Test Set)
# ============================================================
print("=" * 60)
print("  QUICK TEMPORAL VALIDATION")
print("=" * 60)

test_deltas = []
test_emb_list = []
test_pids_used = []

for pid in pids_test:
    if pid not in trajectory_embs:
        continue
    seqs = longitudinal[pid]
    v_first = fast_vol_lookup(pid, seqs[0][0])
    v_last  = fast_vol_lookup(pid, seqs[-1][0])
    if v_first is None or v_last is None or v_first < 1.0:
        continue
    delta = (v_last - v_first) / v_first
    test_deltas.append(delta)
    test_emb_list.append(trajectory_embs[pid])
    test_pids_used.append(pid)

if len(test_deltas) >= 10:
    test_deltas = np.array(test_deltas)
    test_embs_arr = np.stack(test_emb_list)

    # Spearman: trajectory norm vs |volume change|
    norms = np.linalg.norm(test_embs_arr, axis=1)
    rho_norm, p_rho = spearmanr(norms, np.abs(test_deltas))

    # Spearman: predicted progression direction
    # Use first principal component of trajectory embeddings
    from sklearn.decomposition import PCA
    pca = PCA(n_components=1)
    pc1 = pca.fit_transform(test_embs_arr).squeeze()
    rho_pc1, p_pc1 = spearmanr(pc1, test_deltas)
    tau_pc1, p_tau = kendalltau(pc1, test_deltas)

    print(f"\n  Test patients with volume data: {len(test_deltas)}")
    print(f"\n  Trajectory PC1 vs volume change:")
    print(f"    Spearman rho = {rho_pc1:.3f}  (p={p_pc1:.4f})")
    print(f"    Kendall  tau = {tau_pc1:.3f}  (p={p_tau:.4f})")

    # Patient cohort breakdown
    progressive = test_deltas > 0.25
    stable = np.abs(test_deltas) <= 0.25
    responding = test_deltas < -0.25

    print(f"\n  Test cohort:")
    print(f"    Progressive (>25% growth):   {progressive.sum()}")
    print(f"    Stable:                      {stable.sum()}")
    print(f"    Responding (<25% shrink):    {responding.sum()}")

    # Cohen's d between progressive vs stable
    if progressive.sum() >= 3 and stable.sum() >= 3:
        prog_e = test_embs_arr[progressive]
        stab_e = test_embs_arr[stable]
        mean_diff = np.linalg.norm(prog_e.mean(0) - stab_e.mean(0))
        pooled_std = np.sqrt((prog_e.var(0) + stab_e.var(0)).mean())
        cohens_d = mean_diff / (pooled_std + 1e-8)
        print(f"\n  Progressive vs Stable separation:")
        print(f"    Cohen's d = {cohens_d:.3f}  (target > 0.5)")

    # Compare to static embedding performance
    print(f"\n  Comparison to static hybrid embedding (from B1):")
    print(f"    Static T1_spearman_wt  = 0.094")
    print(f"    TaViT  T1 (PC1 rho)   = {rho_pc1:.3f}  {'IMPROVED' if abs(rho_pc1) > 0.094 else 'same'}")
    print(f"    Static T8_kendall_tau  = 0.354")
    print(f"    TaViT  T8 (PC1 tau)   = {tau_pc1:.3f}  {'IMPROVED' if abs(tau_pc1) > 0.354 else 'same'}")
else:
    print(f"  Not enough test patients with volume data ({len(test_deltas)})")

print(f"\n{'=' * 60}")
print(f"  TaViT TRAINING COMPLETE")
print(f"{'=' * 60}")
print(f"\n  Outputs: {OUTPUT_ROOT}")
print(f"  Next: Run Phase4_A2_TaViT_Eval for full temporal evaluation")
